# 📚 Chatbot RAG con LangChain + Azure OpenAI

**Reto proyecto — Desarrollo de Soluciones IA**

Chatbot que aplica **RAG (Retrieval-Augmented Generation)** con **LangChain** para responder preguntas basándose en documentos markdown de una empresa ficticia. Usa **`OpenAIEmbeddings`** con **`text-embedding-3-small`**, almacena los vectores en **`InMemoryVectorStore`**, recupera los fragmentos relevantes por similitud y genera la respuesta con un modelo de chat (**gpt-4o**), manteniendo el contexto de la conversación.

### Mapeo de la estructura de archivos pedida → celdas

```
├── main.py                  → Celda «Interfaz CLI» (run_cli)
├── documents/
│   ├── documento1.md        → se crean en la celda «Documentos markdown»
│   └── documento2.md
├── core/
│   ├── rag_system.py        → Celda «Sistema RAG» (RAGSystem)
│   └── chatbot.py           → Celda «Chatbot»
├── requirements.txt         → Celda de dependencias
├── .env                     → Credenciales (privadas)
└── README.md                → Este encabezado
```

### Configuración (credenciales del lab de Azure)

Crea un archivo **`.env`** con la clave del lab:

```text
AZURE_OPENAI_API_KEY=pega_aqui_la_clave_del_lab
AZURE_OPENAI_ENDPOINT=https://marcvancutseme7172-2656-resource.services.ai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT=gpt-5
AZURE_OPENAI_EMBED_DEPLOYMENT=text-embedding-3-small
```

> ⚠️ **Importante sobre embeddings.** El reto exige `text-embedding-3-small`. Para que funcione, el recurso de Azure debe tener **un deployment con ese nombre**. La lista de modelos del lab (gpt-4o, gpt-4o-mini, gpt-5.2-chat, gpt-5.4-mini) **no incluía embeddings**; si al indexar obtienes un **404**, pide que desplieguen `text-embedding-3-small` en el recurso, o configura `EMBED_API_KEY`/`EMBED_BASE_URL` en el `.env` para usar una clave de OpenAI estándar solo para los embeddings. El modelo de chat seguiría en Azure.

**No subas el `.env`** al entregar; basta con el código.


## 1. Dependencias (`requirements.txt`)

```text
langchain
langchain-openai
langchain-community
python-dotenv
```

> `langchain-core` y `langchain-text-splitters` se instalan automáticamente como dependencias de `langchain`.


In [1]:
# Instalación de dependencias (ejecutar una vez)
%pip install -q langchain langchain-openai langchain-community python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración (`.env`)

Cargamos las credenciales de Azure. Construimos el `base_url` del endpoint v1 (que usan tanto el chat como los embeddings) y normalizamos el endpoint para no duplicar el path. Los embeddings pueden usar credenciales propias (`EMBED_*`) por si hay que apuntarlos a otro recurso.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# --- Endpoint base de Azure (normalizado, sin /openai ni /openai/v1) ---
_raw = os.getenv(
    "AZURE_OPENAI_ENDPOINT",
    "https://marcvancutseme7172-2656-resource.services.ai.azure.com/",
)
_base = _raw.rstrip("/")
for _suf in ("/openai/v1", "/openai"):
    if _base.endswith(_suf):
        _base = _base[: -len(_suf)]
BASE_URL = _base + "/openai/v1/"          # endpoint v1 (compatible con el cliente OpenAI)
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

# Deployments (en Azure, el "model" es el nombre del deployment)
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-5")
EMBED_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBED_DEPLOYMENT", "text-embedding-3-small")

# Credenciales específicas para embeddings (por si hay que usar otro recurso/clave).
# Por defecto, las mismas que el chat.
EMBED_BASE_URL = os.getenv("EMBED_BASE_URL", BASE_URL)
EMBED_API_KEY = os.getenv("EMBED_API_KEY", API_KEY)

if API_KEY:
    print("✅ Configuración cargada.")
    print(f"   Chat:       '{CHAT_DEPLOYMENT}'  @ {BASE_URL}")
    print(f"   Embeddings: '{EMBED_DEPLOYMENT}' @ {EMBED_BASE_URL}")
else:
    print("⚠️  Falta AZURE_OPENAI_API_KEY. Crea un .env con la clave del lab.")


✅ Configuración cargada.
   Chat:       'gpt-5'  @ https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/
   Embeddings: 'text-embedding-3-small' @ https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/


## 3. Documentos markdown (`documents/`)

Creamos dos documentos ficticios de la empresa **NeoraTech Soluciones** (cada uno con más de 500 palabras): información general y políticas internas. Esta celda los escribe en la carpeta `documents/`.


In [3]:
import os

os.makedirs("documents", exist_ok=True)

DOCUMENTO1 = """# NeoraTech Soluciones — Información General

## Historia
NeoraTech Soluciones es una empresa tecnológica ficticia fundada en 2014 en Valencia, España, por un grupo de
ingenieros de software y especialistas en datos. Lo que empezó como una pequeña consultora de tres personas en un
espacio de coworking se ha convertido en una compañía con más de 250 empleados y oficinas en Valencia, Madrid y
Lisboa. Durante sus primeros años, NeoraTech se especializó en el desarrollo de aplicaciones web a medida para pymes,
pero pronto amplió su catálogo hacia la inteligencia artificial, la analítica de datos y la automatización de procesos.

## Misión
Nuestra misión es ayudar a las organizaciones a tomar mejores decisiones mediante soluciones tecnológicas accesibles,
fiables y centradas en las personas. Creemos que la tecnología debe simplificar el trabajo diario, no complicarlo, y
trabajamos para que cada proyecto genere un impacto medible en nuestros clientes.

## Visión
Aspiramos a ser, en 2030, el socio tecnológico de referencia en el sur de Europa para proyectos de inteligencia
artificial aplicada, reconocidos por la calidad de nuestro trabajo, la transparencia con los clientes y el cuidado de
nuestro equipo.

## Valores
- **Cercanía:** acompañamos a cada cliente como si su proyecto fuera nuestro.
- **Excelencia:** cuidamos los detalles y medimos resultados.
- **Transparencia:** comunicamos de forma honesta los avances y los riesgos.
- **Aprendizaje continuo:** dedicamos tiempo a formarnos y experimentar.

## Servicios
NeoraTech ofrece cuatro grandes líneas de servicio:

1. **Desarrollo de software a medida.** Diseñamos y construimos aplicaciones web y móviles adaptadas a las necesidades
   de cada cliente, con especial atención a la usabilidad y el rendimiento.
2. **Inteligencia artificial y machine learning.** Desarrollamos modelos de predicción, sistemas de recomendación,
   chatbots y soluciones de visión por computador. Acompañamos al cliente desde la prueba de concepto hasta la puesta
   en producción.
3. **Analítica de datos.** Ayudamos a las empresas a centralizar, limpiar y visualizar sus datos mediante cuadros de
   mando interactivos que facilitan la toma de decisiones.
4. **Consultoría y formación.** Ofrecemos auditorías tecnológicas y programas de formación para que los equipos de
   nuestros clientes adquieran autonomía.

## Sectores
Trabajamos principalmente con clientes de los sectores retail, logística, educación y salud. Entre nuestros proyectos
más destacados se encuentran un sistema de previsión de demanda para una cadena de supermercados, una plataforma de
teleformación para una universidad privada y un asistente virtual para una aseguradora.

## Metodología de trabajo
En NeoraTech aplicamos metodologías ágiles, principalmente Scrum y Kanban, adaptadas al tamaño de cada proyecto.
Trabajamos en ciclos cortos de dos semanas que terminan con una demostración al cliente, de forma que el avance sea
visible y se puedan incorporar cambios cuanto antes. Cada equipo cuenta con un responsable técnico, un gestor de
proyecto y varios desarrolladores o analistas. Damos mucha importancia a la revisión de código entre compañeros y a la
automatización de pruebas, porque consideramos que la calidad no es negociable.

## Tecnologías
Nuestro equipo domina un amplio abanico de tecnologías. En desarrollo web utilizamos principalmente Python, JavaScript
y TypeScript, con frameworks como FastAPI, Django y React. Para la parte de datos e inteligencia artificial empleamos
bibliotecas como PyTorch, scikit-learn y LangChain, además de servicios en la nube de Azure y AWS. Para la analítica y
la visualización trabajamos con Power BI y herramientas a medida basadas en bibliotecas de gráficos interactivos.

## Equipo y cultura
El equipo de NeoraTech está formado por más de 250 personas de perfiles muy diversos: desarrolladores, científicos de
datos, diseñadores de experiencia de usuario, gestores de proyecto y personal de administración. Cuidamos especialmente
el ambiente de trabajo y la conciliación, porque creemos que un equipo motivado entrega mejores resultados. Cada nuevo
empleado pasa por un plan de incorporación de dos semanas con un mentor asignado que le acompaña en sus primeros
proyectos.

## Compromiso
NeoraTech mantiene un firme compromiso con la sostenibilidad y la diversidad. Compensamos la huella de carbono de
nuestras oficinas, fomentamos el teletrabajo para reducir desplazamientos y aplicamos procesos de selección que
garantizan la igualdad de oportunidades. Cada año destinamos parte de nuestros beneficios a proyectos sociales
relacionados con la alfabetización digital en comunidades con pocos recursos.
"""

DOCUMENTO2 = """# NeoraTech Soluciones — Políticas y Procedimientos Internos

## Horario laboral
La jornada habitual en NeoraTech es de 40 horas semanales, de lunes a viernes. Aplicamos un modelo de **horario
flexible**: la entrada puede realizarse entre las 7:30 y las 9:30, y la salida entre las 16:30 y las 18:30, siempre que
se cumplan las horas diarias y la franja de presencia común de 10:00 a 13:00. Los viernes la jornada es intensiva y
finaliza a las 15:00.

## Teletrabajo
NeoraTech opera con un modelo **híbrido**. Cada empleado puede trabajar hasta tres días por semana en remoto, previo
acuerdo con su responsable de equipo. La empresa proporciona el equipo informático necesario y una ayuda mensual para
gastos de conexión a quienes teletrabajan de forma habitual.

## Beneficios
Los empleados de NeoraTech disponen de los siguientes beneficios:
- **Seguro médico privado** gratuito, ampliable a familiares con condiciones especiales.
- **23 días laborables de vacaciones** al año, más los días de asuntos propios establecidos por convenio.
- **Presupuesto de formación** anual de 600 euros por persona para cursos, libros o congresos.
- **Plan de retribución flexible** que incluye tickets restaurante, guardería y transporte.
- **Día libre de cumpleaños**, que puede disfrutarse en la fecha que el empleado prefiera dentro del mes.

## Código de conducta
Todos los miembros de NeoraTech se comprometen a mantener un trato respetuoso y profesional. No se tolera ningún tipo
de acoso, discriminación o comportamiento que atente contra la dignidad de las personas. La comunicación debe ser
honesta y constructiva, y los conflictos deben resolverse a través del diálogo o, si es necesario, con la mediación del
departamento de Personas.

## Confidencialidad
La información de clientes y proyectos es estrictamente confidencial. Los empleados no deben compartir datos internos,
código fuente ni documentación con terceros sin autorización expresa. El incumplimiento de esta política puede dar
lugar a medidas disciplinarias.

## Procedimiento de solicitud de vacaciones
Las vacaciones se solicitan a través del portal interno del empleado con un mínimo de **15 días de antelación**. El
responsable de equipo dispone de cinco días laborables para aprobarlas o proponer fechas alternativas en caso de
solapamiento con otros compañeros. Las vacaciones del periodo estival deben solicitarse antes del 30 de abril.

## Procedimiento de gastos
Los gastos relacionados con el trabajo (desplazamientos, dietas, material) se reportan mensualmente adjuntando los
justificantes en el portal de gastos. El plazo máximo para presentar un gasto es de 30 días desde su realización. El
reembolso se efectúa junto con la nómina del mes siguiente.

## Formación y desarrollo profesional
NeoraTech apuesta por el crecimiento de su equipo. Además del presupuesto anual de formación, cada empleado define con
su responsable un plan de desarrollo individual que se revisa dos veces al año. La empresa organiza sesiones internas de
intercambio de conocimiento ("NeoraTalks") cada quince días, donde los compañeros comparten aprendizajes de sus
proyectos. También se financia la asistencia a un congreso del sector al año para quienes lo soliciten con antelación.

## Política de igualdad y diversidad
NeoraTech garantiza la igualdad de trato y oportunidades entre todas las personas, con independencia de su género,
origen, edad, orientación o creencias. Existe un plan de igualdad que se revisa anualmente y una comisión encargada de
velar por su cumplimiento. Los procesos de selección y promoción se basan exclusivamente en criterios objetivos de
mérito y capacidad.

## Canal de denuncias
La empresa dispone de un canal de denuncias confidencial para comunicar cualquier conducta contraria al código de
conducta o a la legislación vigente. Las comunicaciones pueden realizarse de forma anónima y se gestionan con total
garantía de confidencialidad y sin represalias para quien las presenta de buena fe.

## Seguridad y bienestar
NeoraTech promueve hábitos saludables: ofrece fruta en las oficinas, organiza sesiones de pausa activa y dispone de un
programa de apoyo psicológico confidencial. En materia de seguridad informática, es obligatorio el uso del gestor de
contraseñas corporativo y la autenticación en dos pasos en todas las herramientas internas.
"""

with open("documents/documento1.md", "w", encoding="utf-8") as f:
    f.write(DOCUMENTO1)
with open("documents/documento2.md", "w", encoding="utf-8") as f:
    f.write(DOCUMENTO2)

print("✅ Documentos creados en 'documents/':")
for nombre in ("documento1.md", "documento2.md"):
    ruta = os.path.join("documents", nombre)
    palabras = len(open(ruta, encoding="utf-8").read().split())
    print(f"   - {nombre}  ({palabras} palabras)")


✅ Documentos creados en 'documents/':
   - documento1.md  (686 palabras)
   - documento2.md  (660 palabras)


## 4. Sistema RAG (`core/rag_system.py`)

`RAGSystem` se encarga de la parte de **recuperación**: carga los `.md`, los **divide en fragmentos** con `RecursiveCharacterTextSplitter`, crea los **embeddings** con `OpenAIEmbeddings` (`text-embedding-3-small`) y los guarda en un **`InMemoryVectorStore`**. El método `recuperar()` hace la **búsqueda por similitud** y devuelve los fragmentos más relevantes para una consulta.


In [4]:
import glob
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# El enunciado exige este modelo de embeddings concreto. Se fija aquí y NO se toma
# de variables de entorno, para que una mala configuración no rompa el requisito.
MODELO_EMBEDDINGS_REQUERIDO = "text-embedding-3-small"


class RAGSystem:
    """Gestiona embeddings, vector store y recuperación de fragmentos."""

    def __init__(self, carpeta: str = "documents", k: int = 4,
                 base_url: str | None = None, api_key: str | None = None,
                 chunk_size: int = 800, chunk_overlap: int = 120):
        self.carpeta = carpeta
        self.modelo_emb = MODELO_EMBEDDINGS_REQUERIDO
        self.k = k
        self.embeddings = OpenAIEmbeddings(
            model=MODELO_EMBEDDINGS_REQUERIDO, base_url=base_url, api_key=api_key
        )
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.vector_store: InMemoryVectorStore | None = None
        self.info_fragmentos: dict[str, int] = {}
        self.num_documentos: int = 0

    def cargar_documentos(self) -> list[Document]:
        """Lee los .md de la carpeta y los convierte en Documentos."""
        rutas = sorted(glob.glob(os.path.join(self.carpeta, "*.md")))
        if not rutas:
            raise FileNotFoundError(f"No se encontraron documentos .md en '{self.carpeta}'.")
        documentos = []
        for ruta in rutas:
            with open(ruta, encoding="utf-8") as f:
                texto = f.read()
            documentos.append(Document(page_content=texto, metadata={"source": os.path.basename(ruta)}))
        return documentos

    def indexar(self) -> int:
        """Divide, vectoriza y almacena los documentos. Devuelve el nº de fragmentos."""
        documentos = self.cargar_documentos()
        self.num_documentos = len(documentos)
        # El enunciado pide procesar AL MENOS dos documentos
        if self.num_documentos < 2:
            print(f"⚠️  Solo se encontró {self.num_documentos} documento .md en '{self.carpeta}'. "
                  "El enunciado requiere al menos 2. Revisa que se hayan creado documento1.md y documento2.md.")
        fragmentos = self.splitter.split_documents(documentos)
        self.info_fragmentos = {}
        for frag in fragmentos:
            fuente = frag.metadata.get("source", "?")
            self.info_fragmentos[fuente] = self.info_fragmentos.get(fuente, 0) + 1
        self.vector_store = InMemoryVectorStore(self.embeddings)
        self.vector_store.add_documents(fragmentos)
        return len(fragmentos)

    def recuperar(self, consulta: str, k: int | None = None,
                  umbral: float | None = None) -> list[Document]:
        """Búsqueda por similitud. Devuelve los k fragmentos más relevantes.

        umbral: si se indica, descarta los fragmentos por debajo de ese valor.
            IMPORTANTE sobre la semántica del score: `InMemoryVectorStore` devuelve una
            puntuación de **similitud coseno** (cuanto MAYOR, más relevante; típicamente en
            torno a 0..1). Por eso aquí se conservan los fragmentos con `score >= umbral`
            (p. ej. umbral=0.3 descarta los poco relevantes). Ten en cuenta que otros vector
            stores (FAISS, Chroma...) pueden devolver una DISTANCIA (cuanto menor, mejor); si
            cambias de backend, deberás invertir la comparación.
        """
        if self.vector_store is None:
            raise RuntimeError("El índice no está construido. Llama primero a indexar().")
        if umbral is None:
            return self.vector_store.similarity_search(consulta, k=k or self.k)
        pares = self.vector_store.similarity_search_with_score(consulta, k=k or self.k)
        return [doc for doc, score in pares if score >= umbral]

    def recuperar_con_puntuaciones(self, consulta: str, k: int | None = None):
        """Devuelve pares (Documento, puntuación de similitud) para analizar la relevancia."""
        if self.vector_store is None:
            raise RuntimeError("El índice no está construido. Llama primero a indexar().")
        return self.vector_store.similarity_search_with_score(consulta, k=k or self.k)


C:\Users\macdu\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Chatbot conversacional (`core/chatbot.py`)

`Chatbot` integra el `RAGSystem` con el modelo de chat (`ChatOpenAI`, deployment `gpt-4o`). Implementa el flujo **consulta → retrieval → generación → respuesta** y mantiene el **historial** de la conversación. El prompt de sistema obliga a responder **solo** con la información recuperada de los documentos.


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

RESPUESTA_SIN_INFO = "No dispongo de esa información en los documentos."

SYSTEM_PROMPT = (
    "Eres el asistente virtual de la empresa NeoraTech Soluciones. Respondes SIEMPRE en español.\n"
    "Responde ÚNICAMENTE con la información del CONTEXTO proporcionado (extraído de los documentos "
    "internos). NO uses conocimiento externo ni inventes datos.\n"
    "Si el contexto no contiene la respuesta, NO intentes deducirla ni completarla con conocimiento "
    f"general: di con claridad '{RESPUESTA_SIN_INFO}'\n"
    "Ejemplos del comportamiento esperado:\n"
    "- Pregunta: '¿Cuántos días de vacaciones hay?' y el contexto menciona '23 días laborables' "
    "→ respondes con ese dato tomado del contexto.\n"
    "- Pregunta: '¿Quién ganó la Champions en 2010?' (no aparece en el contexto) → respondes que no "
    "dispones de esa información en los documentos, SIN inventarla.\n"
    "Sé claro y conciso, y aprovecha el historial para mantener la coherencia de la conversación."
)


class Chatbot:
    """Chatbot RAG: combina recuperación de documentos con generación de respuestas."""

    def __init__(self, rag: "RAGSystem", modelo: str = "gpt-5",
                 base_url: str | None = None, api_key: str | None = None,
                 temperature: float = 0.2, max_context_chars: int = 6000,
                 max_turnos_historial: int = 10, umbral_similitud: float | None = None):
        self.rag = rag
        self.llm = ChatOpenAI(model=modelo, base_url=base_url, api_key=api_key, temperature=temperature)
        self.max_context_chars = max_context_chars
        self.max_turnos_historial = max_turnos_historial
        self.umbral_similitud = umbral_similitud
        self.historial: list = []

    def _construir_contexto(self, fragmentos) -> str:
        partes, total = [], 0
        for d in fragmentos:
            trozo = f"[{d.metadata.get('source', '?')}] {d.page_content}"
            if total + len(trozo) > self.max_context_chars and partes:
                break
            partes.append(trozo)
            total += len(trozo)
        return "\n\n".join(partes)

    def responder(self, consulta: str):
        """Flujo RAG completo. Devuelve (respuesta, fragmentos). respuesta=None si hay error."""
        # 1) Retrieval (errores diferenciados)
        try:
            fragmentos = self.rag.recuperar(consulta, umbral=self.umbral_similitud)
        except Exception as e:
            print(f"⚠️  Error en la recuperación (retrieval) ({type(e).__name__}: {e}).")
            return None, []

        contexto = self._construir_contexto(fragmentos)

        # Atajo: si no hay contexto útil, respondemos con el mensaje estándar SIN llamar al modelo
        if not contexto.strip():
            self.historial.append(HumanMessage(content=consulta))
            self.historial.append(AIMessage(content=RESPUESTA_SIN_INFO))
            return RESPUESTA_SIN_INFO, fragmentos

        # 2) Generación: solo se envían los últimos N turnos del historial
        historial_reciente = self.historial[-2 * self.max_turnos_historial:]
        mensajes = (
            [SystemMessage(content=SYSTEM_PROMPT)]
            + historial_reciente
            + [HumanMessage(content=f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {consulta}")]
        )
        try:
            respuesta = self.llm.invoke(mensajes)
        except Exception as e:
            print(f"⚠️  Error en el modelo de chat (generación) ({type(e).__name__}: {e}).")
            return None, fragmentos

        texto = respuesta.content
        self.historial.append(HumanMessage(content=consulta))
        self.historial.append(AIMessage(content=texto))
        return texto, fragmentos

    def reiniciar(self) -> None:
        self.historial.clear()


## 6. Construir el índice (embeddings + vector store)

Creamos el `RAGSystem`, indexamos los documentos (aquí se generan los embeddings y se llenan los vectores) y preparamos el `Chatbot`. **Esta celda hace llamadas a la API de embeddings**, así que requiere `text-embedding-3-small` disponible en el recurso.


In [6]:
rag = None
bot = None

if not API_KEY:
    print("❌ Falta AZURE_OPENAI_API_KEY. Configura el .env y reejecuta la celda 4.")
else:
    try:
        rag = RAGSystem(carpeta="documents", k=4, base_url=EMBED_BASE_URL, api_key=EMBED_API_KEY)

        # Garantía estricta del requisito de embeddings
        if rag.modelo_emb != MODELO_EMBEDDINGS_REQUERIDO:
            raise RuntimeError(
                f"El modelo de embeddings debe ser '{MODELO_EMBEDDINGS_REQUERIDO}', "
                f"pero es '{rag.modelo_emb}'."
            )
        if EMBED_DEPLOYMENT != MODELO_EMBEDDINGS_REQUERIDO:
            print(f"⚠️  En Azure el deployment de embeddings debería llamarse "
                  f"'{MODELO_EMBEDDINGS_REQUERIDO}' (tu .env indica '{EMBED_DEPLOYMENT}').")

        # Comprobación previa: el enunciado pide al menos 2 documentos .md distintos
        n_docs = len(rag.cargar_documentos())
        if n_docs < 2:
            print(f"⚠️  Se ha encontrado solo {n_docs} documento .md en 'documents/'. "
                  "El enunciado requiere al menos 2 (documento1.md y documento2.md). "
                  "Ejecuta la celda que crea los documentos.")
        else:
            print(f"📄 Documentos encontrados: {n_docs}.")

        print("⏳ Indexando documentos (generando embeddings)...")
        n = rag.indexar()
        bot = Chatbot(rag, modelo=CHAT_DEPLOYMENT, base_url=BASE_URL, api_key=API_KEY)
        print(f"✅ Índice listo: {n} fragmentos vectorizados (modelo: {rag.modelo_emb}).")
        for fuente, cantidad in sorted(rag.info_fragmentos.items()):
            print(f"   • {fuente}: {cantidad} fragmentos")
        print("Chatbot preparado.")
    except Exception as e:
        nombre = type(e).__name__
        if "NotFound" in nombre or "404" in str(e):
            print(f"❌ No se encontró el deployment de embeddings '{MODELO_EMBEDDINGS_REQUERIDO}' (404). "
                  "Asegúrate de que existe en el recurso o usa EMBED_API_KEY/EMBED_BASE_URL.")
        elif "Authentication" in nombre:
            print("❌ Credenciales inválidas (401). Revisa AZURE_OPENAI_API_KEY.")
        elif "Connection" in nombre:
            print("❌ No se pudo conectar con el endpoint. Revisa AZURE_OPENAI_ENDPOINT y tu conexión.")
        else:
            print(f"❌ Error al construir el índice: {nombre}: {e}")


📄 Documentos encontrados: 2.
⏳ Indexando documentos (generando embeddings)...
✅ Índice listo: 16 fragmentos vectorizados (modelo: text-embedding-3-small).
   • documento1.md: 9 fragmentos
   • documento2.md: 7 fragmentos
Chatbot preparado.


## 7. Interfaz CLI (`main.py`)

Bucle de conversación. Construye el sistema RAG si no se le pasa uno ya preparado, muestra mensajes informativos, recupera y responde, y permite salir con `/salir`. Maneja errores básicos de conexión y del modelo.


In [7]:
def inicializar_sistema(k: int = 4) -> "Chatbot":
    """Construye e indexa el RAGSystem y devuelve un Chatbot listo (como haría main.py)."""
    if not API_KEY:
        raise RuntimeError("Falta AZURE_OPENAI_API_KEY. Configura el .env.")
    sistema = RAGSystem(carpeta="documents", k=k, base_url=EMBED_BASE_URL, api_key=EMBED_API_KEY)
    n = sistema.indexar()
    print(f"✅ Índice listo ({n} fragmentos de {sistema.num_documentos} documentos).")
    return Chatbot(sistema, modelo=CHAT_DEPLOYMENT, base_url=BASE_URL, api_key=API_KEY)


def _ayuda_cli() -> None:
    print("\nℹ️  Soy el asistente RAG de NeoraTech. Respondo preguntas sobre la empresa "
          "(historia, misión, servicios, políticas, horarios, beneficios...) usando SOLO "
          "la información de sus documentos internos.")
    print("Comandos:  /ayuda (esta ayuda)  ·  /reiniciar (borra el historial)  ·  /salir (terminar)")
    print("Ejemplos:  '¿Cuál es la misión?'  ·  '¿Cuántos días de vacaciones tengo?'\n")


def run_cli(chatbot: "Chatbot | None" = None, mostrar_fuentes: bool = True) -> None:
    """Interfaz por terminal del chatbot RAG."""
    print("=" * 62)
    print("📚 Chatbot RAG — NeoraTech Soluciones (LangChain + Azure OpenAI)")
    print("=" * 62)

    # Inicialización delegada en una función reutilizable
    if chatbot is None:
        try:
            print("⏳ Cargando documentos y construyendo el índice...")
            chatbot = inicializar_sistema()
        except Exception as e:
            print(f"❌ No se pudo inicializar el sistema RAG ({type(e).__name__}: {e}).")
            return

    _ayuda_cli()

    while True:
        try:
            consulta = input("🧑 Tú > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 ¡Hasta luego!")
            break

        if not consulta:
            continue
        comando = consulta.lower()
        if comando in ("/salir", "/exit", "/quit", "salir", "quit"):
            print("👋 ¡Hasta luego!")
            break
        if comando in ("/ayuda", "/help"):
            _ayuda_cli()
            continue
        if comando in ("/reiniciar", "/reset"):
            chatbot.reiniciar()
            print("🔄 Historial de conversación borrado.")
            continue

        respuesta, fragmentos = chatbot.responder(consulta)
        if respuesta is None:
            print("❗ No se pudo completar la consulta por un problema técnico. Inténtalo de nuevo.\n")
            continue

        print(f"\n🤖 {respuesta}")
        if mostrar_fuentes and fragmentos:
            fuentes = sorted({d.metadata.get("source", "?") for d in fragmentos})
            print(f"   📎 Fuentes: {', '.join(fuentes)}\n")


### ▶️ Opción A — Preguntar sin `input()` (recomendado en notebooks)

Reutiliza el `bot` ya construido en la celda 6 y hazle preguntas directamente. El historial se mantiene entre llamadas.


In [8]:
if bot is None:
    print("⚠️  Ejecuta antes la celda 6 para construir el índice (necesita embeddings disponibles).")
else:
    for pregunta in [
        "¿Cuál es la misión de la empresa?",
        "¿Cuántos días de vacaciones tengo y con cuánta antelación debo pedirlas?",
    ]:
        print(f"🧑 {pregunta}")
        # Inspección de pertinencia: qué fragmentos se recuperan y con qué puntuación
        print("   🔎 Fragmentos recuperados (fuente · score):")
        for doc, score in rag.recuperar_con_puntuaciones(pregunta):
            inicio = doc.page_content.strip().replace("\n", " ")[:60]
            print(f"      - {doc.metadata.get('source', '?')} · {score:.3f} · «{inicio}...»")
        respuesta, fragmentos = bot.responder(pregunta)
        print(f"   🤖 {respuesta}\n")


🧑 ¿Cuál es la misión de la empresa?
   🔎 Fragmentos recuperados (fuente · score):
      - documento1.md · 0.637 · «## Misión Nuestra misión es ayudar a las organizaciones a to...»
      - documento1.md · 0.470 · «1. **Desarrollo de software a medida.** Diseñamos y construi...»
      - documento1.md · 0.468 · «## Compromiso NeoraTech mantiene un firme compromiso con la ...»
      - documento1.md · 0.416 · «## Valores - **Cercanía:** acompañamos a cada cliente como s...»
   🤖 Nuestra misión es ayudar a las organizaciones a tomar mejores decisiones mediante soluciones tecnológicas accesibles, fiables y centradas en las personas. Creemos que la tecnología debe simplificar el trabajo diario, no complicarlo, y trabajamos para que cada proyecto genere un impacto medible en nuestros clientes.

🧑 ¿Cuántos días de vacaciones tengo y con cuánta antelación debo pedirlas?
   🔎 Fragmentos recuperados (fuente · score):
      - documento2.md · 0.669 · «## Procedimiento de solicitud de vacaciones Las v

### ▶️ Opción B — CLI interactiva con `run_cli()`

Lanza el bucle interactivo. Reutiliza el `bot` ya indexado para no volver a generar embeddings.


In [9]:
run_cli(chatbot=bot)   # pasa el bot ya construido; si es None, run_cli lo construye solo


📚 Chatbot RAG — NeoraTech Soluciones (LangChain + Azure OpenAI)

ℹ️  Soy el asistente RAG de NeoraTech. Respondo preguntas sobre la empresa (historia, misión, servicios, políticas, horarios, beneficios...) usando SOLO la información de sus documentos internos.
Comandos:  /ayuda (esta ayuda)  ·  /reiniciar (borra el historial)  ·  /salir (terminar)
Ejemplos:  '¿Cuál es la misión?'  ·  '¿Cuántos días de vacaciones tengo?'


🤖 Tienes 23 días laborables de vacaciones al año.
   📎 Fuentes: documento2.md


🤖 Nuestra misión es ayudar a las organizaciones a tomar mejores decisiones mediante soluciones tecnológicas accesibles, fiables y centradas en las personas. Creemos que la tecnología debe simplificar el trabajo diario, no complicarlo, y trabajamos para que cada proyecto genere un impacto medible en nuestros clientes.
   📎 Fuentes: documento1.md, documento2.md

🔄 Historial de conversación borrado.
👋 ¡Hasta luego!
